# UQTS-2026: Alpha Discovery Lab
## Phase 1: Signal vs. Fluid Framework

This notebook demonstrates the foundational components of the **UQTS-2026** research lab:
1. **Bi-temporal Data Engine**: Point-in-Time (PIT) consistency.
2. **Fractional Differentiation**: Stationarity with memory preservation.
3. **Wavelet Spectrograms**: Multi-resolution spectral analysis.
4. **Residualized Alpha**: Idiosyncratic target labeling.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from research_lab.data_engine import DataEngine
from research_lab.alpha_core import FractionalDifferencer, WaveletFeatureGenerator
from research_lab.alpha_labeler import AlphaLabeler

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [12, 8]

### 1. Data Ingestion (Bi-temporal PIT)
We generate synthetic data with a 'Knowledge Time' delay to verify zero look-ahead bias.

In [ ]:
engine = DataEngine()
tickers = ['AAPL', 'MSFT', 'SPY']
engine.generate_synthetic_pit_data(tickers, days=500)

# Fetch a PIT view as of midway through the period
as_of_date = datetime(2020, 1, 1) + timedelta(days=250)
pit_view_aapl = engine.get_pit_view('AAPL', as_of_date)
pit_view_spy = engine.get_pit_view('SPY', as_of_date)

print(f"Data points known for AAPL as of {as_of_date}: {len(pit_view_aapl)}")
pit_view_aapl.tail()

### 2. Fractional Differentiation ($d=0.4$)
Stationarizing the series while preserving long-term memory.

In [ ]:
fd = FractionalDifferencer(d=0.4)
aapl_stationary = fd.transform(pit_view_aapl['close'])

fig, ax1 = plt.subplots()

ax1.set_xlabel('Event Time')
ax1.set_ylabel('Original Price', color='tab:blue')
ax1.plot(pit_view_aapl['event_time'], pit_view_aapl['close'], color='tab:blue', label='Original')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Stationary (d=0.4)', color='tab:red')
ax2.plot(pit_view_aapl['event_time'], aapl_stationary, color='tab:red', label='Stationary')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('AAPL: Original vs. Fractionally Differenced')
fig.tight_layout()
plt.show()

### 3. Wavelet Spectrogram (Morlet)
Analyzing the signal across dyadic scales to detect regimes.

In [ ]:
wfg = WaveletFeatureGenerator()
spectrogram = wfg.generate(aapl_stationary)

plt.imshow(spectrogram, aspect='auto', cmap='jet', extent=[0, len(aapl_stationary), 1, 8])
plt.colorbar(label='Amplitude')
plt.title('Market Spectrogram: AAPL (Dyadic Scales 2^1 to 2^8)')
plt.ylabel('Log Scale')
plt.xlabel('Time Steps')
plt.show()

### 4. Residualized Alpha Generation
Extracting idiosyncratic alpha by removing market beta.

In [ ]:
labeler = AlphaLabeler()

# Calculate returns
aapl_returns = pit_view_aapl['close'].pct_change().dropna().values
spy_returns = pit_view_spy['close'].pct_change().dropna().values

# Ensure alignment
min_len = min(len(aapl_returns), len(spy_returns))
aapl_returns = aapl_returns[:min_len]
spy_returns = spy_returns[:min_len]

alpha_residuals = labeler.residualize(aapl_returns, spy_returns)

plt.scatter(spy_returns, aapl_returns, alpha=0.5, label='Raw Returns')
plt.scatter(spy_returns, alpha_residuals, alpha=0.5, color='red', label='Residualized Alpha')
plt.axhline(0, color='black', lw=1)
plt.axvline(0, color='black', lw=1)
plt.xlabel('Market Returns (SPY)')
plt.ylabel('Asset Returns / Alpha')
plt.legend()
plt.title('Idiosyncratic Alpha extraction via Residualization')
plt.show()

print(f"Correlation (Raw): {np.corrcoef(aapl_returns, spy_returns)[0,1]:.4f}")
print(f"Correlation (Alpha): {np.corrcoef(alpha_residuals, spy_returns)[0,1]:.4f}")